# Whisky clustering with Groovy

The data-science core of the classic BeakerX-era `Whiskey.ipynb` from
[groovy-data-science](https://github.com/paulk-asert/groovy-data-science),
ported to the `groovy-jupyter` kernel: `%%classpath add mvn` becomes `@Grab`
(session-wide), `%import` becomes plain imports (they persist across cells),
and session state uses binding-style variables.

CSV parsing uses `groovy-csv`'s `CsvSlurper` (new in Groovy 6, shipped with
the kernel — no grab needed). Smile is pinned to 1.5.3 — the last
Apache-2.0-licensed release. 86 Scotch distilleries, 12 flavor dimensions,
clustered with k-means. Expect cluster sizes of 25/44/17 (cluster numbering
may permute between runs).

The dataset (`whiskey.csv`, alongside this notebook) is the classic 86-distillery
flavor study, copied from groovy-data-science so the example is self-contained
(original source: [niss.org ScotchWhisky01.txt](https://www.niss.org/sites/default/files/ScotchWhisky01.txt)
— see `whiskey_source.txt`). The kernel runs in the notebook's directory, so a
relative path suffices.

In [6]:
@Grab('com.github.haifengl:smile-core:1.5.3')
import groovy.csv.CsvSlurper
import smile.clustering.KMeans
'deps ready'

deps ready

In [7]:
rows = new CsvSlurper().parse(new File('whiskey.csv'))
"${rows.size()} rows X ${rows[0].size()} cols"

86 rows X 14 cols

In [8]:
features = rows[0].keySet() - ['RowID', 'Distillery']
data = rows.collect { row ->
    features.collect { f -> row[f] as double } as double[]
} as double[][]
[data.length, data[0].length]

[86, 12]

In [9]:
kmeans = KMeans.lloyd(data, 3)
clusters = kmeans.clusterLabel.toList()
clusters.countBy { it }

{2=32, 1=38, 0=16}

In [10]:
names = rows*.Distillery
grouped = (0..<clusters.size()).groupBy { clusters[it] }
grouped.collectEntries { k, v -> [k, v.take(5).collect { names[it] }] }

{2=[Aberfeldy, Aberlour, Ardmore, Auchroisk, Balmenach], 1=[AnCnoc, ArranIsleOf, Auchentoshan, Aultmore, Benriach], 0=[Ardbeg, Balblair, Bowmore, Bruichladdich, Caol Ila]}